In [2]:
import requests
import pandas as pd

cities = {
    "Bakı": (40.4093, 49.8671),
    "Gəncə": (40.6828, 46.3606),
    "Sumqayıt": (40.5897, 49.6686),
    "Lənkəran": (38.7543, 48.8506),
    "Mingəçevir": (40.7703, 47.0489),
    "Şəki": (41.1919, 47.1706),
    "Naxçıvan": (39.2089, 45.4122)
}

all_weather_data = {}

for city, (latitude, longitude) in cities.items():

    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": "2026-01-01",
        "end_date": "2026-06-30",
        "daily": [
            "temperature_2m_mean",
            "precipitation_sum",
            "wind_speed_10m_max"
        ],
        "timezone": "Asia/Baku"
    }

    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()

    weather_data = response.json()

    all_weather_data[city] = weather_data

    print(city, ":", len(weather_data["daily"]["time"]), "gün")

print("\nAPI-dən məlumatlar uğurla çəkildi!")

Bakı : 181 gün
Gəncə : 181 gün
Sumqayıt : 181 gün
Lənkəran : 181 gün
Mingəçevir : 181 gün
Şəki : 181 gün
Naxçıvan : 181 gün

API-dən məlumatlar uğurla çəkildi!


In [1]:
import json

with open(PROJECT_ROOT / "data/api/weather_7_cities_2026_01_06.json", encoding="utf-8") as f:
    all_weather_data = json.load(f)

NameError: name 'PROJECT_ROOT' is not defined

In [3]:
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

API_DIR = PROJECT_ROOT / "data" / "api"
API_DIR.mkdir(parents=True, exist_ok=True)

json_path = API_DIR / "weather_7_cities_2026_01_06.json"

with open(json_path, "w", encoding="utf-8") as file:
    json.dump(
        all_weather_data,
        file,
        ensure_ascii=False,
        indent=4
    )

print("JSON faylı saxlanıldı:", json_path)
print("Şəhər sayı:", len(all_weather_data))


JSON faylı saxlanıldı: /workspaces/Bank_Analysis/data/api/weather_7_cities_2026_01_06.json
Şəhər sayı: 7


In [4]:
import sys
import pandas as pd
from sqlalchemy import text

sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))

from db import get_engine

# JSON faylını oxuyuruq
with open(json_path, "r", encoding="utf-8") as file:
    saved_weather_data = json.load(file)

weather_frames = []

for city, weather_data in saved_weather_data.items():

    city_df = pd.DataFrame(weather_data["daily"])
    city_df["city"] = city

    weather_frames.append(city_df)

weather_all_df = pd.concat(
    weather_frames,
    ignore_index=True
)

print("SQL-ə göndəriləcək sətir sayı:", len(weather_all_df))

# Məlumatların tamlığını yoxlayırıq
assert len(weather_all_df) == 1267, "Sətir sayı gözləniləndən fərqlidir!"

assert not weather_all_df.duplicated(
    subset=["city", "time"]
).any(), "Təkrarlanan şəhər-tarix qeydləri var!"

# SQL Server-ə yazırıq
engine = get_engine()

weather_all_df.to_sql(
    name="weather",
    con=engine,
    schema="bronze",
    if_exists="replace",
    index=False
)

# Nəticəni yoxlayırıq
with engine.connect() as conn:
    result = pd.read_sql(
        text("""
            SELECT city, COUNT(*) AS row_count
            FROM bronze.weather
            GROUP BY city
            ORDER BY city
        """),
        conn
    )

display(result)

print("Ümumi sətir sayı:", result["row_count"].sum())

SQL-ə göndəriləcək sətir sayı: 1267


,city,row_count
0,Baki,181
1,G?nc?,181
2,L?nk?ran,181
3,Ming?çevir,181
4,Naxçivan,181
5,S?ki,181
6,Sumqayit,181


Ümumi sətir sayı: 1267


In [8]:
calendar_df = pd.read_csv(
    PROJECT_ROOT / "data" / "bank_data" / "calendar_reference.csv"
)

print("Sətir və sütun sayı:", calendar_df.shape)
display(calendar_df.head())

Sətir və sütun sayı: (181, 6)


,date,is_holiday,holiday_name,is_weekend,is_workday,is_pre_holiday
0,1/1/2026,1,Yeni il,0,0,0
1,1/2/2026,1,Yeni il,0,0,0
2,1/3/2026,0,NaN,1,0,0
3,1/4/2026,0,NaN,1,0,0
4,1/5/2026,0,NaN,0,1,0


In [9]:
from sqlalchemy import text

sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
from db import get_engine

engine = get_engine()

calendar_df.to_sql(
    name="calendar_reference",
    con=engine,
    schema="bronze",
    if_exists="replace",
    index=False
)

with engine.connect() as conn:
    count = conn.execute(
        text("SELECT COUNT(*) FROM bronze.calendar_reference")
    ).scalar()

print("bronze.calendar_reference:", count, "sətir")

bronze.calendar_reference: 181 sətir


Indi məzənnə məlumatlarını api ilə çəkək

In [15]:
import requests
import pandas as pd

url = "https://api.frankfurter.dev/v2/rates"

params = {
    "from": "2026-01-01",
    "to": "2026-06-30",
    "base": "AZN",
    "quotes": "USD,EUR"
}

response = requests.get(url, params=params, timeout=60)

print("HTTP status:", response.status_code)

if response.ok:
    fx_data = response.json()
    fx_df = pd.DataFrame(fx_data)

    print("Sətir və sütun sayı:", fx_df.shape)
    display(fx_df.head())
else:
    print("API cavabı:", response.text[:500])

HTTP status: 200
Sətir və sütun sayı: (362, 4)


,date,base,quote,rate
0,2026-01-01,AZN,EUR,0.50084
1,2026-01-01,AZN,USD,0.58837
2,2026-01-02,AZN,EUR,0.50095
3,2026-01-02,AZN,USD,0.58830
4,2026-01-03,AZN,EUR,0.50108


In [16]:
from pathlib import Path

API_DIR = PROJECT_ROOT / "data" / "api"
API_DIR.mkdir(parents=True, exist_ok=True)

file_path = API_DIR / "exchange_rates_2026_01_06.csv"

fx_df.to_csv(file_path, index=False, encoding="utf-8")

print("Fayl saxlanıldı:", file_path)
print("Sətir sayı:", len(fx_df))

Fayl saxlanıldı: /workspaces/Bank_Analysis/data/api/exchange_rates_2026_01_06.csv
Sətir sayı: 362


In [17]:
import sys
from sqlalchemy import text

sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
from db import get_engine

engine = get_engine()

fx_df.to_sql(
    name="exchange_rates",
    con=engine,
    schema="bronze",
    if_exists="replace",
    index=False
)

with engine.connect() as conn:
    count = conn.execute(
        text("SELECT COUNT(*) FROM bronze.exchange_rates")
    ).scalar()

print("bronze.exchange_rates:", count, "sətir")

bronze.exchange_rates: 362 sətir


In [18]:
from sqlalchemy import text

with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT
            TABLE_SCHEMA,
            TABLE_NAME
        FROM INFORMATION_SCHEMA.TABLES
        WHERE TABLE_SCHEMA = 'bronze'
        ORDER BY TABLE_NAME
    """))

    for row in result:
        print(f"{row.TABLE_SCHEMA}.{row.TABLE_NAME}")

bronze.calendar_reference
bronze.customers
bronze.exchange_rates
bronze.loans
bronze.transactions
bronze.weather


In [1]:
cities = {
    "Bakı": (40.4093, 49.8671),
    "Gəncə": (40.6828, 46.3606),
    "Sumqayıt": (40.5897, 49.6686),
    "Lənkəran": (38.7543, 48.8506),
    "Mingəçevir": (40.7703, 47.0489),
    "Şəki": (41.1919, 47.1706),
    "Naxçıvan": (39.2089, 45.4122)
}

print("Şəhər sayı:", len(cities))

for city, coordinates in cities.items():
    print(city, coordinates)

Şəhər sayı: 7
Bakı (40.4093, 49.8671)
Gəncə (40.6828, 46.3606)
Sumqayıt (40.5897, 49.6686)
Lənkəran (38.7543, 48.8506)
Mingəçevir (40.7703, 47.0489)
Şəki (41.1919, 47.1706)
Naxçıvan (39.2089, 45.4122)
